In [203]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import floor
import torch
import torch.nn as nn

In [204]:
def torch_sigmoid(x):
    return torch.sigmoid(x)

def torch_relu(x):
    return torch.relu(x)

def torch_leaky_relu(x, alpha=0.01):
    return torch.nn.functional.leaky_relu(x, negative_slope=alpha)

def torch_softmax(x):
    return torch.softmax(x, dim=1)

In [205]:
class LinCell(nn.Module):
    def __init__(self, inp_dim, out_dim):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(inp_dim, out_dim) * np.sqrt(2.0 / inp_dim))
        self.bias = nn.Parameter(torch.zeros(1, out_dim))
        self.input_dim = inp_dim
        self.output_dim = out_dim

    def forward(self, x):
        return x @ self.weights + self.bias
    
class ActivationCell(nn.Module):
    def __init__(self, activation_func):
        super().__init__()
        self.activation = activation_func

    def forward(self, x):
        return self.activation(x)
    
class NeuralNet(nn.Module):
    def __init__(self, layer_types, inp_dims, out_dims):
        super().__init__()
        self.layer_types = layer_types
        self.inp_dims = inp_dims
        self.out_dims = out_dims
        self.layers = nn.ModuleList()
        idx = 0
        for ltype in layer_types:
            ltype_lower = ltype.lower()
            if ltype_lower == "linear":
                self.layers.append(LinCell(inp_dims[idx], out_dims[idx]))
                idx += 1
            elif ltype_lower == "relu":
                self.layers.append(ActivationCell(torch_relu))
            elif ltype_lower == "sigmoid":
                self.layers.append(ActivationCell(torch_sigmoid))
            elif ltype_lower == "tanh":
                self.layers.append(ActivationCell(torch.tanh))
            elif ltype_lower == "leakyrelu":
                self.layers.append(ActivationCell(lambda x: torch_leaky_relu(x)))
            elif ltype_lower == "softmax":
                self.layers.append(ActivationCell(lambda x: torch_softmax(x)))
            else:
                raise ValueError(f"Unsupported layer type: {ltype}")

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def save(self, filename="neural.pth"):
        checkpoint = {
            'layer_types': self.layer_types,
            'inp_dims': self.inp_dims,
            'out_dims': self.out_dims,
            'state_dict': self.state_dict()
        }
        torch.save(checkpoint, filename)

    def load(self, filename="neuraln.pth"):
        checkpoint = torch.load(filename, map_location='cpu')
        if (checkpoint['layer_types'] != self.layer_types or
            checkpoint['inp_dims'] != self.inp_dims or
            checkpoint['out_dims'] != self.out_dims):
            print("警告：加载的模型结构与当前网络不完全一致，可能出错。")
        self.load_state_dict(checkpoint['state_dict'])

    def meanSquareError(self, X, Y):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        if not isinstance(Y, torch.Tensor):
            Y = torch.tensor(Y, dtype=torch.float32)
        was_training = self.training
        self.training = False
        with torch.no_grad():
            pred = self.forward(X)
            loss = torch.mean((pred - Y) ** 2)
        self.training = was_training
        return loss.item()

    def train(self, X, Y, learning_rate):
        if not hasattr(self, 'optimizer'):
            self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        else:
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = learning_rate

        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        if not isinstance(Y, torch.Tensor):
            Y = torch.tensor(Y, dtype=torch.float32)

        self.training = True
        self.optimizer.zero_grad()
        pred = self.forward(X)
        loss = torch.mean((pred - Y) ** 2)
        loss.backward()
        self.optimizer.step()
        return loss.item()

提取数据

In [206]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")
feature_cols = ["np","nn","delta","I", "betaMinusQ", "betaMinusQUncertainty", "halflife_log_sec_uncertainty"]
#feature_cols = ["np", "na", "betaMinusQ", "betaMinusQUncertainty"]
target_cols = ["halflife_log_sec"]

trainX = train_df[feature_cols].values
trainY = train_df[target_cols].values

validX = valid_df[feature_cols].values
validY = valid_df[target_cols].values

testX = test_df[feature_cols].values
testY = test_df[target_cols].values

输入归一化

In [207]:
#核子数类：np(第0列), na(第1列) 除以 118
trainX = trainX.copy().astype(float)
validX = validX.copy().astype(float)
testX = testX.copy().astype(float)

trainX[:, 0] /= 118.0   # np
trainX[:, 1] /= 118.0   # na
validX[:, 0] /= 118.0
validX[:, 1] /= 118.0
testX[:, 0] /= 118.0
testX[:, 1] /= 118.0

#其余特征（从第2列开始）除以三个集合的最大值
#拼接三个集合，计算每列最大值
other_cols_start = 2   # 假设第0列np，第1列na，其余从第2列起
all_other = np.vstack([trainX[:, other_cols_start:],
                       validX[:, other_cols_start:],
                       testX[:, other_cols_start:]])
max_vals = np.max(all_other, axis=0)

#对各集合的其余特征除以对应最大值
trainX[:, other_cols_start:] /= max_vals
validX[:, other_cols_start:] /= max_vals
testX[:, other_cols_start:] /= max_vals

建立神经网络

In [208]:
NNeurons = 32
NLayers = 10
dim_Input =len(feature_cols)
dim_Target = len(target_cols)
inp = [dim_Input]
opt = [NNeurons]
types = ["linear","leakyrelu"]
for i in range(NLayers-2):
    types.append("linear")
    types.append("leakyrelu")
    inp.append(NNeurons)
    opt.append(NNeurons)
types.append("linear")
inp.append(NNeurons)
opt.append(dim_Target)
neural=NeuralNet(types,inp,opt)

训练

In [209]:
mode = int(input("输出模式，0为从零训练，1为继续训练，2为读取数据并拟合"))

epsilon = 0.03#定义容许误差 Определение допустимой погрешности
maxTime = int(1E5)
alpha = 1E-3
best_lo_validation = 1E20
maxPatience = 50
patience = maxPatience#如果验证集误差持续不降低则失去耐心停止训练
best_weights = []
best_biases = []
timeTrained=0

trainX_t = torch.tensor(trainX, dtype=torch.float32)
trainY_t = torch.tensor(trainY, dtype=torch.float32)
validX_t = torch.tensor(validX, dtype=torch.float32)
validY_t = torch.tensor(validY, dtype=torch.float32)
testX_t  = torch.tensor(testX,  dtype=torch.float32)
testY_t  = torch.tensor(testY,  dtype=torch.float32)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(neural.parameters(), lr=alpha)

if mode:
    neural.load()

if mode == 1 or mode == 0:
    for i in range(maxTime):
        lo = neural.train(trainX, trainY, alpha)
        timeTrained += 1
        lo_validation = neural.meanSquareError(validX, validY)
        if best_lo_validation > lo_validation:
            best_lo_validation = lo_validation
            best_state_dict = neural.state_dict()
            patience = maxPatience
        else:
            patience -= 1
        if lo < epsilon or patience < 0:
            print(lo)
            break
    # 恢复最佳参数
    if best_state_dict is not None:
        neural.load_state_dict(best_state_dict)
    neural.save()
'''
if mode == 1 or mode == 0:
    print(f"Количество итераций обучения: {timeTrained}")
    print(f"Среднеквадратичная ошибка после обучения: {neural.meanSquareError(trainX, trainY)}")
else:
    print(f"Среднеквадратичная ошибка для обучающей выборки: {neural.meanSquareError(trainX, trainY)}")

print(f"Среднеквадратичная ошибка для тестовой выборки: {neural.meanSquareError(testX, testY)}")
print(f"Среднеквадратичная ошибка для валидационной выборки: {neural.meanSquareError(validX, validY)}")'''
if mode == 1 or mode == 0:
    print(f"训练次数: {timeTrained}")
    print(f"训练后均方差: {neural.meanSquareError(trainX, trainY)}")
else:
    print(f"训练集均方差: {neural.meanSquareError(trainX, trainY)}")

print(f"测试集均方差: {neural.meanSquareError(testX, testY)}")
print(f"验证集均方差: {neural.meanSquareError(validX, validY)}")

1.7114980220794678
训练次数: 211
训练后均方差: 1.6925034523010254
测试集均方差: 1.4677014350891113
验证集均方差: 2.535447835922241
